<a href="https://colab.research.google.com/github/victorbaraunaAcad/crise-saude-ufam/blob/main/coleta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json, time, datetime, pathlib, unicodedata
from getpass import getpass

import requests
import pandas as pd

print(f"pandas  {pd.__version__}")
print(f"requests {requests.__version__}")

pandas  2.2.3
requests 2.32.4


In [2]:
USUARIO = "victorbaraunaAcad"      # trocar pelo seu usuário
REPO    = "crise-saude-ufam" # trocar pelo nome do repositório

token = getpass("Cole o token do GitHub (fine-grained, Contents: Read/write): ")

!git clone https://{token}@github.com/{USUARIO}/{REPO}.git
%cd {REPO}
!git config user.name "victorbaraunaAcad"
!git config user.email "victor.barauna@icomp.ufam.edu.br"

RAIZ     = pathlib.Path.cwd()
BRUTOS   = RAIZ / "dados_brutos"
TRATADOS = RAIZ / "dados_tratados"
BRUTOS.mkdir(exist_ok=True); TRATADOS.mkdir(exist_ok=True)
print("Trabalhando em:", RAIZ)

Cole o token do GitHub (fine-grained, Contents: Read/write): ··········
Cloning into 'crise-saude-ufam'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 22 (delta 7), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 8.38 KiB | 8.38 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/crise-saude-ufam
Trabalhando em: /content/crise-saude-ufam


In [3]:
ARQ_PROV = RAIZ / "proveniencia.csv"

def registrar(fonte, url, metodo, parametros, n_linhas, arquivo_bruto):
    linha = pd.DataFrame([{
        "fonte": fonte, "url": url, "metodo": metodo,
        "parametros": json.dumps(parametros, ensure_ascii=False),
        "n_linhas": n_linhas,
        "arquivo_bruto": str(pathlib.Path(arquivo_bruto).relative_to(RAIZ)),
        "coletado_em": datetime.datetime.now().astimezone().isoformat(timespec="seconds"),
    }])
    cabecalho = not ARQ_PROV.exists() or ARQ_PROV.stat().st_size == 0
    linha.to_csv(ARQ_PROV, mode="a", header=cabecalho, index=False)
    print(f"  ✓ proveniência: {fonte} ({n_linhas} linhas)")

def normalizar(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return s.upper().strip()

In [4]:
CODIGOS_UF = {"AM": 13, "RS": 43}
CIDADES_CENTRO_OESTE = {"Cuiabá": 51, "Goiânia": 52, "Brasília": 53}

def buscar_municipios_uf(cod_uf, sigla):
    url = f"https://servicodados.ibge.gov.br/api/v1/localidades/estados/{cod_uf}/municipios"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    bruto = r.json()

    arq = BRUTOS / f"ibge_localidades_{sigla}.json"
    arq.write_text(json.dumps(bruto, ensure_ascii=False, indent=2), encoding="utf-8")
    registrar(f"IBGE Localidades — {sigla}", url, "API REST (GET, JSON)",
              {"uf": cod_uf}, len(bruto), arq)

    return pd.json_normalize(bruto).rename(columns={
        "id": "codigo_ibge", "nome": "nome_municipio",
        "microrregiao.mesorregiao.UF.sigla": "uf",
    })[["codigo_ibge", "nome_municipio", "uf"]]

partes = []

# AM e RS inteiros
for sigla, cod_uf in CODIGOS_UF.items():
    partes.append(buscar_municipios_uf(cod_uf, sigla))

# Cuiabá, Goiânia, Brasília — filtradas de dentro do estado de cada uma
for cidade, cod_uf in CIDADES_CENTRO_OESTE.items():
    df_uf = buscar_municipios_uf(cod_uf, f"CO_{cod_uf}")
    df_uf["nome_norm"] = df_uf["nome_municipio"].map(normalizar)
    achado = df_uf[df_uf["nome_norm"] == normalizar(cidade)]
    if achado.empty:
        print(f"  ✗ não achei {cidade} — confira o nome retornado pela API")
    partes.append(achado.drop(columns="nome_norm"))

municipios = pd.concat(partes, ignore_index=True).drop_duplicates("codigo_ibge")
municipios["codigo_ibge"] = municipios["codigo_ibge"].astype("int64")
municipios["nome_norm"] = municipios["nome_municipio"].map(normalizar)

print(municipios.groupby("uf").size())
municipios.head()

  ✓ proveniência: IBGE Localidades — AM (62 linhas)
  ✓ proveniência: IBGE Localidades — RS (497 linhas)
  ✓ proveniência: IBGE Localidades — CO_51 (142 linhas)
  ✓ proveniência: IBGE Localidades — CO_52 (246 linhas)
  ✓ proveniência: IBGE Localidades — CO_53 (1 linhas)
uf
AM     62
DF      1
GO      1
MT      1
RS    497
dtype: int64


,codigo_ibge,nome_municipio,uf,nome_norm
0,1300029,Alvarães,AM,ALVARAES
1,1300060,Amaturá,AM,AMATURA
2,1300086,Anamã,AM,ANAMA
3,1300102,Anori,AM,ANORI
4,1300144,Apuí,AM,APUI


In [5]:
print("Duplicatas:", municipios["codigo_ibge"].duplicated().sum())      # precisa ser 0
print("Total de linhas:", len(municipios))                              # esperado: ~62+497+3 = 562
print("Dígitos do código:", municipios["codigo_ibge"].astype(str).str.len().unique())  # precisa ser [7]
municipios[municipios["uf"].isin(["MT","GO","DF"])]                     # confirma as 3 cidades certas

Duplicatas: 0
Total de linhas: 562
Dígitos do código: [7]


,codigo_ibge,nome_municipio,uf,nome_norm
559,5103403,Cuiabá,MT,CUIABA
560,5208707,Goiânia,GO,GOIANIA
561,5300108,Brasília,DF,BRASILIA


In [6]:
!git add -A
!git commit -m "Fonte 1: municipios AM inteiro + RS inteiro + Cuiaba/Goiania/Brasilia"
!git push

[main 780e4c0] Fonte 1: municipios AM inteiro + RS inteiro + Cuiaba/Goiania/Brasilia
 6 files changed, 37918 insertions(+)
 create mode 100644 dados_brutos/ibge_localidades_AM.json
 create mode 100644 dados_brutos/ibge_localidades_CO_51.json
 create mode 100644 dados_brutos/ibge_localidades_CO_52.json
 create mode 100644 dados_brutos/ibge_localidades_CO_53.json
 create mode 100644 dados_brutos/ibge_localidades_RS.json
remote: Permission to victorbaraunaAcad/crise-saude-ufam.git denied to victorbaraunaAcad.
fatal: unable to access 'https://github.com/victorbaraunaAcad/crise-saude-ufam.git/': The requested URL returned error: 403
